# 01 — Árbol basado en C4.5: categorías, números y prepoda

Implementación propia de la teoría de **SEM04-DecisionTrees, filminas 23–30**.
La construcción, las fórmulas y el recorrido están escritos explícitamente aquí;
pandas organiza los datos, no aprende el árbol. La extensión numérica sigue
también a **Liu (2011), Web Data Mining, 2.ª ed., pp. 75–76**.

**Alcance:** clasificación con atributos categóricos, numéricos y mixtos, sin
faltantes; selección local por Gain Ratio y prepoda por ganancia mínima.
No implementa pospoda, ponderación de faltantes ni todas las heurísticas del
C4.5 original. La visualización y los controles interactivos quedan para otra HU.

**Contrato:** `predict(X)` recibe un DataFrame y devuelve una columna `prediccion`
con el mismo índice y orden. `fit(X, y)` recibe `y` como DataFrame con la columna
`objetivo`; es una decisión interna de este modelo, no un contrato adicional
exigido a Random Forest.

Ejecutar todas las celdas desde la raíz del repositorio o desde `notebooks/`.
En Colab se necesita el repositorio completo y sus dependencias; las fuentes de
UCI se cargan localmente. `datos_tp.py` reutiliza la preparación de la HU anterior;
scikit-learn allí solo carga Iris y separa train/test.

In [1]:
from collections import Counter
from dataclasses import dataclass, field
from math import fsum, isfinite, log2
from numbers import Real

import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype

## 1. Cálculos de impureza y ganancia

Para las clases $c$, $p_c$ es su frecuencia relativa en el nodo $D$.
Para cada rama $D_j$, $q_j=|D_j|/|D|$.

$$H(D)=-\sum_c p_c\log_2 p_c$$
$$H_A(D)=\sum_j q_jH(D_j)$$
$$Gain(D,A)=H(D)-H_A(D)$$
$$SplitInfo(D,A)=-\sum_j q_j\log_2 q_j$$
$$GainRatio(D,A)=\frac{Gain(D,A)}{SplitInfo(D,A)}$$

Corresponden a las filminas **25, 26, 28 y 29** y a Liu pp. 72–75.
`entropia` representa `impurityEval-1` y `entropia_ponderada` representa
`impurityEval-2`. La ganancia y el denominador se utilizan realmente en la
selección; no son funciones decorativas. `fsum` solo suma términos numéricos.

Convenios: $0\log_2 0=0$; entropía del conjunto vacío igual a 0. Una partición
con Split Info igual a 0 no es elegible. El valor defensivo 0 devuelto por
`gain_ratio` en ese caso no implica que el cociente matemático esté definido.

In [2]:
def entropia(etiquetas):
    """Filmina 25: H(D) = -sum(p_c * log2(p_c)), en bits.

    Solo se recorren clases presentes: los términos 0*log2(0) valen 0.
    Para un conjunto vacío se devuelve 0 por convenio.
    """
    conteos = Counter(etiquetas)
    total = sum(conteos.values())
    if total == 0:
        return 0.0
    return -fsum((n / total) * log2(n / total) for n in conteos.values())


def entropia_ponderada(grupos):
    """Filmina 26: sum(|D_j| / |D| * H(D_j)). Grupos de etiquetas."""
    total = sum(len(grupo) for grupo in grupos)
    if total == 0:
        return 0.0
    return fsum(len(grupo) / total * entropia(grupo) for grupo in grupos)


def ganancia_informacion(etiquetas, grupos):
    """Filmina 28: Gain(D, A) = H(D) - H_A(D).

    Precondición: grupos es una partición de etiquetas, sin perder ni repetir casos.
    """
    return entropia(etiquetas) - entropia_ponderada(grupos)


def split_info(grupos):
    """Denominador de la filmina 29: entropía de los tamaños de las ramas."""
    total = sum(len(grupo) for grupo in grupos)
    if total == 0:
        return 0.0
    return -fsum(
        (len(grupo) / total) * log2(len(grupo) / total)
        for grupo in grupos if len(grupo) > 0
    )


def gain_ratio(etiquetas, grupos):
    """Filmina 29: Gain(D, A) / SplitInfo(D, A).

    El cociente con denominador 0 no está definido. Se devuelve 0 como
    convenio defensivo; el árbol DESCARTA esa partición antes de seleccionarla.
    """
    denominador = split_info(grupos)
    if denominador == 0:
        return 0.0
    return ganancia_informacion(etiquetas, grupos) / denominador


def evaluar_particion(etiquetas, indices_por_rama):
    """Evalúa una partición usando posiciones de filas, no sus índices pandas."""
    grupos = [[etiquetas[i] for i in posiciones]
              for posiciones in indices_por_rama.values()]
    return {
        "entropia_ponderada": entropia_ponderada(grupos),
        "ganancia": ganancia_informacion(etiquetas, grupos),
        "split_info": split_info(grupos),
        "gain_ratio": gain_ratio(etiquetas, grupos),
        "ramas": indices_por_rama,
    }

## 2. Particiones categóricas y búsqueda de cortes numéricos

**Categorías:** una rama por valor observado (filmina 23). Un atributo constante
no divide el nodo. **Números:** ordenar los valores distintos y evaluar cada
valor salvo el mayor como corte `t`; se crean `A ≤ t` y `A > t` (filmina 30;
detalle de búsqueda en Liu pp. 75–76). Se usa el valor observado inferior,
no el punto medio. Valores repetidos permanecen juntos.

Se busca el mayor Gain Ratio entre candidatos **del nodo actual**, sin usar
test. Es una selección local y voraz; no garantiza el árbol global óptimo.
En empates exactos se conserva el menor corte. Los atributos se compararán
en el orden de las columnas de entrenamiento.

In [3]:
def particion_categorica(valores, etiquetas):
    """Una rama por valor OBSERVADO; orden de primera aparición."""
    ramas = {}
    for posicion, valor in enumerate(valores):
        ramas.setdefault(valor, []).append(posicion)
    if len(ramas) < 2:
        return None  # Atributo constante: Split Info = 0, no separa casos.
    return evaluar_particion(etiquetas, ramas)


def buscar_mejor_umbral(valores, etiquetas):
    """Filmina 30 y Liu pp. 75-76: evaluar los r-1 cortes distintos.

    Para cada valor v_i salvo el mayor: izquierda A <= v_i, derecha A > v_i.
    Se elige el mayor Gain Ratio local; un empate conserva el menor corte.
    No se discretiza globalmente ni se consultan los datos de prueba.
    """
    valores, etiquetas = list(valores), list(etiquetas)
    if len(valores) != len(etiquetas):
        raise ValueError("Cada valor debe tener su etiqueta correspondiente.")
    if any(not isinstance(v, Real) or isinstance(v, bool) or not isfinite(v)
           for v in valores):
        raise ValueError("Los valores numéricos deben ser reales y finitos.")

    candidatos = sorted(set(valores))[:-1]
    mejor = None
    for umbral in candidatos:
        ramas = {
            "<=": [i for i, valor in enumerate(valores) if valor <= umbral],
            ">": [i for i, valor in enumerate(valores) if valor > umbral],
        }
        # Dos grupos no vacíos: el mayor valor nunca es candidato.
        evaluacion = evaluar_particion(etiquetas, ramas)
        evaluacion["umbral"] = umbral
        if mejor is None or evaluacion["gain_ratio"] > mejor["gain_ratio"]:
            mejor = evaluacion
    return mejor  # None si no hay al menos dos valores distintos.

## 3. Representación de un nodo

Una hoja conserva su clase. Un nodo interno además guarda el atributo, las
ramas y, si corresponde, un corte numérico. Se conservan conteos, métricas y
motivos de parada para inspección y una futura visualización.

**Decisiones de implementación no especificadas en las filminas:** empates de
clase resueltos por primera aparición local; categorías sin rama en predicción
resueltas con la mayoría del nodo alcanzado. No se modifica el árbol al predecir.

In [4]:
@dataclass
class Nodo:
    """Una decisión o una hoja; también conserva datos para explicar el árbol.

    clase es la mayoría local, tanto en hojas como en nodos internos.
    En un empate de clases se conserva la primera observada en ese nodo.
    """
    clase: object
    conteos: dict
    n_ejemplos: int
    entropia: float
    atributo: object = None
    tipo: str | None = None
    umbral: float | int | None = None
    hijos: dict = field(default_factory=dict)
    ganancia: float | None = None
    split_info: float | None = None
    gain_ratio: float | None = None
    motivo_parada: str | None = None

    @property
    def es_hoja(self):
        return not self.hijos

## 4. Construcción recursiva y predicción

Se sigue la filmina 23, incorporando Gain Ratio de la 29 para seleccionar la
partición. La condición de parada compara la **ganancia de información** de esa
partición con `threshold`, en bits. Es prepoda, no pospoda.

Se preserva la desigualdad estricta `ganancia < threshold`. El valor inicial
`threshold=0` es una decisión del proyecto, no un valor indicado en las filminas:
permite dividir con ganancia cero. La recursión termina porque cada división
válida produce subconjuntos más pequeños, y las categorías usadas se retiran.

Los atributos numéricos permanecen disponibles y pueden volver a dividirse.
Las columnas numéricas (excepto booleanos) se tratan como números; texto,
booleanos y columnas pandas `category` se tratan como categorías. Los códigos
categóricos de Heart ya se convierten a texto en la preparación compartida.

También se detiene si los atributos restantes son constantes y no hay una
partición válida. En hojas impuras se usa la clase mayoritaria local.

In [5]:
class ArbolC45:
    """Árbol didáctico basado en las filminas 23-30, con prepoda.

    No delega entrenamiento, criterios de partición ni predicción a una
    biblioteca de ML. No implementa pospoda ni tratamiento de faltantes.
    threshold se aplica a Information Gain, en bits, con comparación estricta <.
    """

    def __init__(self, threshold=0.0):
        self.threshold = threshold
        self.raiz_ = None
        self._validar_threshold()

    def _validar_threshold(self):
        if (not isinstance(self.threshold, Real) or isinstance(self.threshold, bool)
                or not isfinite(self.threshold) or self.threshold < 0):
            raise ValueError("threshold debe ser un número real finito y no negativo.")

    @staticmethod
    def _validar_X(X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un pandas.DataFrame.")
        if X.empty or X.isna().any().any():
            raise ValueError("X debe contener casos y atributos, sin valores faltantes.")
        if not X.columns.is_unique or X.columns.isna().any():
            raise ValueError("Los nombres de los atributos deben ser únicos y no nulos.")
        for columna in X.columns:
            serie = X[columna]
            if is_complex_dtype(serie.dtype):
                raise ValueError("No se admiten atributos de números complejos.")
            if is_numeric_dtype(serie.dtype):
                if not all(isfinite(v) for v in serie.tolist()):
                    raise ValueError("Los atributos numéricos deben contener valores finitos.")
            else:
                # Los valores categóricos serán claves del diccionario de ramas.
                try:
                    for valor in serie:
                        hash(valor)
                except TypeError as error:
                    raise ValueError("Las categorías deben ser valores simples/hashables.") from error

    def fit(self, X, y):
        """Entrena con X y con y DataFrame de una columna llamada objetivo."""
        self._validar_threshold()
        self._validar_X(X)
        if not isinstance(y, pd.DataFrame):
            raise TypeError("y debe ser un pandas.DataFrame.")
        if list(y.columns) != ["objetivo"] or not X.index.equals(y.index):
            raise ValueError("y debe tener la columna objetivo y el mismo índice y orden que X.")
        if y.isna().any().any():
            raise ValueError("y no debe contener valores faltantes.")
        etiquetas = y["objetivo"].tolist()
        try:
            Counter(etiquetas)
        except TypeError as error:
            raise ValueError("Las clases deben ser valores simples/hashables.") from error

        self.columnas_ = X.columns.copy()
        self.tipos_ = {
            columna: ("numerico" if is_numeric_dtype(X[columna].dtype)
                      and not is_bool_dtype(X[columna].dtype) else "categorico")
            for columna in X.columns
        }
        self.raiz_ = self._construir(X, etiquetas, list(X.columns))
        return self

    def _construir(self, X, etiquetas, atributos):
        conteos = dict(Counter(etiquetas))
        clase = max(conteos, key=conteos.get)  # Primer máximo en orden de aparición.
        nodo = Nodo(clase, conteos, len(etiquetas), entropia(etiquetas))

        # Filmina 23, líneas 1-4: clase pura o atributos agotados.
        if len(conteos) == 1:
            nodo.motivo_parada = "clase_pura"
            return nodo
        if not atributos:
            nodo.motivo_parada = "sin_atributos"
            return nodo

        # Líneas 7-11; se usa la normalización Gain Ratio de la filmina 29.
        mejor = None
        for atributo in atributos:
            valores = X[atributo].tolist()
            if self.tipos_[atributo] == "numerico":
                candidata = buscar_mejor_umbral(valores, etiquetas)
            else:
                candidata = particion_categorica(valores, etiquetas)
            if candidata is None or candidata["split_info"] == 0:
                continue
            if mejor is None or candidata["gain_ratio"] > mejor["gain_ratio"]:
                mejor = dict(candidata, atributo=atributo)

        # Puede haber atributos disponibles pero constantes en este nodo.
        if mejor is None:
            nodo.motivo_parada = "sin_particion_valida"
            return nodo

        # Línea 12: conservar '<', no reemplazarlo por '<='.
        # threshold=0 permite una división de ganancia cero (p. ej. XOR).
        if mejor["ganancia"] < self.threshold:
            nodo.motivo_parada = "ganancia_menor_que_threshold"
            return nodo

        # Líneas 15-20: crear ramas y descender sobre cada subconjunto no vacío.
        atributo = mejor["atributo"]
        nodo.atributo = atributo
        nodo.tipo = self.tipos_[atributo]
        nodo.umbral = mejor.get("umbral")
        nodo.ganancia = mejor["ganancia"]
        nodo.split_info = mejor["split_info"]
        nodo.gain_ratio = mejor["gain_ratio"]
        siguientes = ([a for a in atributos if a != atributo]
                      if nodo.tipo == "categorico" else atributos)
        # Para números se conserva el atributo: Liu pp. 75-76 / filmina 30.
        for rama, posiciones in mejor["ramas"].items():
            nodo.hijos[rama] = self._construir(
                X.iloc[posiciones], [etiquetas[i] for i in posiciones], siguientes
            )
        return nodo

    def _predecir_fila(self, fila):
        nodo = self.raiz_
        while not nodo.es_hoja:
            valor = fila[nodo.atributo]
            if nodo.tipo == "numerico":
                rama = "<=" if valor <= nodo.umbral else ">"
            else:
                rama = valor
            if rama not in nodo.hijos:
                # Decisión de implementación: mayoría LOCAL ante categoría no vista.
                return nodo.clase
            nodo = nodo.hijos[rama]
        return nodo.clase

    def predict(self, X: pd.DataFrame) -> pd.DataFrame:
        """Contrato común: una columna prediccion y el mismo índice/orden de X."""
        if self.raiz_ is None:
            raise RuntimeError("El árbol debe entrenarse antes de predecir.")
        self._validar_X(X)
        if not X.columns.equals(self.columnas_):
            raise ValueError("X debe tener las mismas columnas y orden que en entrenamiento.")
        for atributo, tipo in self.tipos_.items():
            if tipo == "numerico" and (
                not is_numeric_dtype(X[atributo].dtype) or is_bool_dtype(X[atributo].dtype)
            ):
                raise ValueError(f"El atributo {atributo!r} debe conservar su tipo numérico.")
        # zip por columnas evita que iterrows convierta enteros en floats.
        predicciones = [self._predecir_fila(dict(zip(self.columnas_, valores)))
                       for valores in X.itertuples(index=False, name=None)]
        return pd.DataFrame({"prediccion": predicciones}, index=X.index.copy())

## 5. Comprobación con el ejemplo de clase

Se transcribe la tabla de **15 casos de la filmina 27**; `ID` es índice, no
atributo. Para `Own_house` hay 6 casos `Yes` en la rama verdadera y 6 `No` +
3 `Yes` en la falsa (filmina 24). El cálculo manual es:

- H(D) = −(6/15) log₂(6/15) − (9/15) log₂(9/15) ≈ **0,970951**.
- H_falsa = −(6/9) log₂(6/9) − (3/9) log₂(3/9) ≈ **0,918296**.
- H_ponderada = (6/15) × 0 + (9/15) × 0,918296 ≈ **0,550978**.
- Gain ≈ **0,419973**; Split Info ≈ **0,970951**; Gain Ratio ≈ **0,432538**.

Las aserciones usan valores de referencia, no una segunda llamada a la función
como resultado esperado. Se verifica además un corte numérico perfecto.

In [6]:
from io import StringIO
from math import isclose
from IPython.display import display

ejemplo_clase = pd.read_csv(StringIO("""ID Age Has_job Own_house Credit_rating objetivo
1 young false false fair No
2 young false false good No
3 young true false good Yes
4 young true true fair Yes
5 young false false fair No
6 middle false false fair No
7 middle false false good No
8 middle true true good Yes
9 middle false true excellent Yes
10 middle false true excellent Yes
11 old false true excellent Yes
12 old false true good Yes
13 old true false good Yes
14 old true false excellent Yes
15 old false false fair No
"""), sep=r"\s+", dtype=str).set_index("ID")
X_clase = ejemplo_clase.drop(columns="objetivo")
y_clase = ejemplo_clase[["objetivo"]]
evaluaciones = []
for atributo in X_clase:
    evaluacion = particion_categorica(X_clase[atributo].tolist(), y_clase["objetivo"].tolist())
    evaluaciones.append({"atributo": atributo, **{k: v for k, v in evaluacion.items() if k != "ramas"}})
tabla_ganancias = pd.DataFrame(evaluaciones).set_index("atributo")
display(tabla_ganancias)

assert isclose(entropia(y_clase["objetivo"]), 0.9709505944546686, abs_tol=1e-12)
for metrica, esperado in {
    "entropia_ponderada": 0.5509775004326937,
    "ganancia": 0.4199730940219749,
    "split_info": 0.9709505944546686,
    "gain_ratio": 0.4325380677663126,
}.items():
    assert isclose(tabla_ganancias.loc["Own_house", metrica], esperado, abs_tol=1e-12)
arbol_clase = ArbolC45().fit(X_clase, y_clase)
assert arbol_clase.raiz_.atributo == "Own_house"
assert arbol_clase.predict(X_clase)["prediccion"].tolist() == y_clase["objetivo"].tolist()

corte = buscar_mejor_umbral([1, 2, 3, 4], ["No", "No", "Yes", "Yes"])
assert corte["umbral"] == 2
assert corte["ganancia"] == corte["split_info"] == corte["gain_ratio"] == 1.0
print("OK: cálculos manuales, ejemplo de clase y corte numérico.")

,entropia_ponderada,ganancia,split_info,gain_ratio
atributo,,,,
Age,0.887943,0.083007,1.584963,0.052372
Has_job,0.647300,0.323650,0.918296,0.352447
Own_house,0.550978,0.419973,0.970951,0.432538
Credit_rating,0.607961,0.362990,1.565596,0.231854


OK: cálculos manuales, ejemplo de clase y corte numérico.


## 6. Datasets y particiones compartidas con Random Forest

Se reutiliza exactamente la carga de Contact Lenses, Iris y Statlog Heart.
La partición sigue siendo 80/20 estratificada, con semilla 42. Los dos modelos
reciben los mismos casos originales; el one-hot sigue siendo interno de RF.

In [7]:
from pathlib import Path
import sys

# Compartir la carga y las mismas particiones sin ejecutar el otro notebook.
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos_tp.py").is_file() and (carpeta / "data/raw").is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError("Ejecutá desde la raíz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from datos_tp import (
    SEMILLA, FRACCION_TEST, TAMANOS, CLASES, cargar_datasets, particionar,
)

datasets = cargar_datasets()
particiones = {nombre: particionar(datos["X"], datos["y"])
               for nombre, datos in datasets.items()}

## 7. Ejecución base en los tres datasets

`THRESHOLD = 0.0` permite modificar la prepoda desde código; el deslizador será
parte de otra HU. Cambiar el parámetro requiere volver a entrenar.

Se muestra accuracy de entrenamiento y prueba como comprobación inicial,
sin exigir un mínimo ni elegir el parámetro mirando test. Para explorar
parametrizaciones después se necesitará validación dentro de entrenamiento.
Contact Lenses tiene solo cinco casos de prueba: cada acierto cambia 20 puntos
porcentuales la accuracy. Estos resultados no prueban superioridad sobre RF.

In [8]:
THRESHOLD = 0.0
modelos = {}
predictores = {}
predicciones = {}
resultados = []
for nombre, parte in particiones.items():
    modelo = ArbolC45(threshold=THRESHOLD).fit(parte["X_train"], parte["y_train"])
    salida_train = modelo.predict(parte["X_train"])
    salida_test = modelo.predict(parte["X_test"])
    modelos[nombre] = modelo
    predictores[nombre] = modelo.predict
    predicciones[nombre] = salida_test
    resultados.append({
        "dataset": nombre, "train": len(parte["X_train"]), "test": len(parte["X_test"]),
        "accuracy_train": (salida_train["prediccion"] == parte["y_train"]["objetivo"]).mean(),
        "accuracy_test": (salida_test["prediccion"] == parte["y_test"]["objetivo"]).mean(),
    })
    print(f"{nombre}: primeras predicciones")
    display(parte["y_test"].join(salida_test).head())
resultados = pd.DataFrame(resultados).set_index("dataset")
display(resultados)

lenses: primeras predicciones


,objetivo,prediccion
id,,
17,sin_lentes,sin_lentes
20,rigidos,rigidos
5,sin_lentes,sin_lentes
14,blandos,blandos
13,sin_lentes,sin_lentes


iris: primeras predicciones


,objetivo,prediccion
38,setosa,setosa
127,virginica,virginica
57,versicolor,versicolor
93,versicolor,versicolor
42,setosa,setosa


heart: primeras predicciones


,objetivo,prediccion
195,ausencia,presencia
132,ausencia,ausencia
162,ausencia,ausencia
129,presencia,presencia
66,ausencia,ausencia


,train,test,accuracy_train,accuracy_test
dataset,,,,
lenses,19,5,1.0,1.000000
iris,120,30,1.0,0.933333
heart,216,54,1.0,0.685185


## 8. Contrato y casos límite

Se comprueba el orden de las predicciones y la ausencia de modificaciones en X.
El caso XOR verifica que una ganancia inicial cero pueda conducir a una buena
división en el siguiente nivel, cuando `threshold=0`. El ejemplo numérico
comprueba que un atributo pueda repetirse en un camino y que `≤` incluya el corte.

La suite `tests/test_c45.py` amplía estas comprobaciones con entradas inválidas,
empates, atributos constantes, agotamiento de categorías, datasets mixtos y
reentrenamiento. Carga las mismas celdas de definiciones: no mantiene otra copia
del algoritmo.

In [9]:
for nombre, parte in particiones.items():
    entrada = parte["X_test"].iloc[::-1].copy(deep=True)
    original = entrada.copy(deep=True)
    salida = predictores[nombre](entrada)
    assert list(salida.columns) == ["prediccion"]
    assert salida.index.equals(entrada.index)
    assert set(salida["prediccion"]) <= set(parte["y_train"]["objetivo"])
    pd.testing.assert_frame_equal(entrada, original)
    pd.testing.assert_frame_equal(salida, predicciones[nombre].iloc[::-1])
    repetida = particionar(datasets[nombre]["X"], datasets[nombre]["y"])
    for clave in parte:
        pd.testing.assert_frame_equal(parte[clave], repetida[clave])

X_xor = pd.DataFrame({"a": ["0", "0", "1", "1"], "b": ["0", "1", "0", "1"]})
y_xor = pd.DataFrame({"objetivo": ["No", "Yes", "Yes", "No"]})
sin_prepoda = ArbolC45(threshold=0).fit(X_xor, y_xor)
assert sin_prepoda.raiz_.ganancia == 0
assert sin_prepoda.predict(X_xor)["prediccion"].tolist() == y_xor["objetivo"].tolist()
assert ArbolC45(threshold=0.01).fit(X_xor, y_xor).raiz_.es_hoja

X_intervalo = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6]})
y_intervalo = pd.DataFrame({"objetivo": ["A", "A", "B", "B", "A", "A"]})
intervalo = ArbolC45().fit(X_intervalo, y_intervalo)
assert intervalo.predict(X_intervalo)["prediccion"].tolist() == y_intervalo["objetivo"].tolist()
assert intervalo.raiz_.atributo == "x"
assert any(hijo.atributo == "x" for hijo in intervalo.raiz_.hijos.values())
print("OK: contrato en tres datasets, prepoda y reutilización de atributos numéricos.")

OK: contrato en tres datasets, prepoda y reutilización de atributos numéricos.


## Fuentes y límites de esta entrega

- **SEM04-DecisionTrees.pptx**, material de clase proporcionado: filmina 23
  (recursión), 24 y 27 (ejemplo), 25–29 (fórmulas), 30 (continuos).
- **Liu, B. (2011). Web Data Mining, 2.ª ed.**, capítulo 3: p. 70 (Fig. 3.4),
  pp. 72–75 (impureza y Gain Ratio), pp. 75–76 (búsqueda y reutilización numérica).
  Copia del libro en `docs/libros/` del repositorio.
- Fuentes y preparación de los datasets: `README.md` y `datos_tp.py`.

La búsqueda de cortes y las fórmulas tienen comprobaciones locales. Ejecutar el
notebook localmente no acredita por sí solo funcionamiento en Google Colab.
Pospoda, visualización, interfaz y comparación experimental de parámetros quedan
pendientes. El contrato compartido sigue siendo solamente `predict(X)`.